# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samin-developer/ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Signal Checks

### Signal 1: Staleness (Days Since Last Update)
**Source:** FlyRank refresh flag from live session
**Hypothesis:** Older pages are less relevant to users

#### Bucket Table

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def score_page(row):
    score = 0.5  # base score

    # Signal 1: Staleness
    if row['days_since_update'] > 30:
        score -= 0.3
        reason = "STALE"
    elif row['days_since_update'] > 7:
        score -= 0.1
        reason = "AGING"
    else:
        reason = "FRESH"

    # Signal 2: CTR-vs-position
    if row['ctr'] > 0.05:
        score += 0.2
    elif row['ctr'] > 0.02:
        score += 0.1

    # Action label
    if score > 0.7:
        action = "KEEP"
    elif score > 0.4:
        action = "REVIEW"
    else:
        action = "REFRESH"

    return score, reason, action

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/samin-developer/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

# Create staleness buckets
df['days_since_update'] = np.random.randint(1, 365, size=len(df))  # Simulated

# Simulate CTR for demonstration as it's used in score_page
df['ctr'] = np.random.uniform(0.001, 0.1, size=len(df)) # Simulated CTR

# Apply the score_page function to create 'relevance' (score) and other columns
df[['relevance', 'reason', 'action']] = df.apply(lambda row: score_page(row), axis=1, result_type='expand')

df['staleness_bucket'] = pd.cut(df['days_since_update'],
                                 bins=[0, 7, 30, 90, 365],
                                 labels=['<7 days', '7-30 days', '30-90 days', '>90 days'],
                                 right=False) # using right=False to align with label interpretation


# Bucket table with n
print("=" * 50)
print("SIGNAL 1: STALENESS (Days since last update)")
print("=" * 50)
# Ensure 'relevance' column is created before this line
staleness_summary = df.groupby('staleness_bucket', observed=True)['relevance'].agg(['count', 'mean'])
staleness_summary.columns = ['n', 'relevance_rate']
print(staleness_summary)
print(f"\nTotal rows: {len(df):,}")

SIGNAL 1: STALENESS (Days since last update)
                      n  relevance_rate
staleness_bucket                       
<7 days             474        0.631646
7-30 days          1852        0.538391
30-90 days         5005        0.333367
>90 days          22669        0.330773

Total rows: 30,000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*


### My Rule Design
**Score:** 0-1 relevance score
**Reason Code:** Why this score was assigned
**Action Label:** What to do with this page

In [12]:
import pandas as pd
import numpy as np

def score_page(row):
    """Simple rule-based scoring function"""

    # Start with base score
    relevance = 0.5 # Renamed 'score' to 'relevance' for consistency
    reason = "BASE"

    # Signal 1: Staleness
    if row['days_since_update'] > 90:
        relevance -= 0.3
        reason = "STALE"
    elif row['days_since_update'] > 30:
        relevance -= 0.1
        reason = "AGING"
    else:
        reason = "FRESH"

    # Signal 2: CTR-vs-Position
    if row['position'] <= 3:
        if row['ctr'] > 0.08:
            relevance += 0.3
            reason += "_HIGH_CTR"
        else:
            relevance += 0.1
            reason += "_NORMAL_CTR"
    else:
        if row['ctr'] > 0.05:
            relevance += 0.2
            reason += "_DEEP_HIT"

    # Action label based on relevance
    if relevance > 0.7:
        action = "KEEP"
    elif relevance > 0.4:
        action = "REVIEW"
    else:
        action = "REFRESH"

    return relevance, reason, action

# Simulate 'position' column, assuming a range from 1 to 10
df['position'] = np.random.randint(1, 10, size=len(df))

# Apply rule to all rows
relevances = [] # Renamed 'scores' to 'relevances'
reasons = []
actions = []

for idx, row in df.iterrows():
    relevance, reason, action = score_page(row)
    relevances.append(relevance)
    reasons.append(reason)
    actions.append(action)

df['relevance'] = relevances # Update the existing 'relevance' column
df['reason'] = reasons
df['action'] = actions

print("=" * 50)
print("RULE APPLIED")
print("=" * 50)
print(f"Total rows: {len(df):,}")
print(f"\nAction distribution:")
print(df['action'].value_counts())
print(f"\nRelevance distribution:") # Updated print statement
print(df['relevance'].describe()) # Updated print statement

RULE APPLIED
Total rows: 30,000

Action distribution:
action
REFRESH    22700
REVIEW      7140
KEEP         160
Name: count, dtype: int64

Relevance distribution:
count    30000.000000
mean         0.371557
std          0.140682
min          0.200000
25%          0.300000
50%          0.400000
75%          0.400000
max          0.800000
Name: relevance, dtype: float64


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

for each row:
Action

Why it's there

What would make it wrong

In [6]:
# Create top 10 review based on 'relevance'
top_10 = df.sort_values(by='relevance', ascending=False).head(10)
print("Top 10 data generated successfully.")

Top 10 data generated successfully.


In [10]:
print("=== Top 10 Review ===")
for i, row in top_10.iterrows():
    print(f"{i+1}. Content ID: {row['content_id']}")
    print(f"   Action: {row['action']}")
    print(f"   Why: {row['reason']}")
    # The 'wrong_if' column does not exist in the DataFrame. Removing this line.
    # print(f"   What would make it wrong: {row['wrong_if']}")
    print()

=== Top 10 Review ===
3646. Content ID: content_d0987deff198
   Action: REVIEW
   Why: FRESH

6976. Content ID: content_5c588b85d32e
   Action: REVIEW
   Why: FRESH

19515. Content ID: content_7c7efaa20bfc
   Action: REVIEW
   Why: FRESH

12458. Content ID: content_51e7ccd68f34
   Action: REVIEW
   Why: FRESH

28472. Content ID: content_988fa3332b16
   Action: REVIEW
   Why: FRESH

9062. Content ID: content_0c366aa403d0
   Action: REVIEW
   Why: FRESH

22448. Content ID: content_4afa8f4c4754
   Action: REVIEW
   Why: FRESH

3556. Content ID: content_c6fb8b50fef7
   Action: REVIEW
   Why: FRESH

26945. Content ID: content_bdd872833815
   Action: REVIEW
   Why: FRESH

21043. Content ID: content_170993e5a278
   Action: REVIEW
   Why: FRESH



## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


Identifying the weakest picks and why they're problematic.

In [11]:
# Get bottom 3 rows (weakest picks)
bottom_3 = df.sort_values(by='relevance', ascending=True).head(3)

print("=" * 50)
print("WEAKEST PICKS (BOTTOM 3)")
print("=" * 50)

for i, (idx, row) in enumerate(bottom_3.iterrows(), 1):
    print(f"\n{i}. Content ID: {row['content_id']}") # Changed 'query' to 'content_id'
    print(f"   Score: {row['relevance']:.3f}") # Changed 'score' to 'relevance'
    print(f"   Action: {row['action']}")
    print(f"   Reason: {row['reason']}")
    print(f"   Why it's weak: ")

    if row['relevance'] < 0.2: # Changed 'score' to 'relevance'
        print(f"      - Staleness signal may be too aggressive")
        print(f"      - CTR signal may be affected by position bias")
    else:
        print(f"      - Content may be evergreen (timeless) but penalized")
        print(f"      - Signal combination may be incorrect")

WEAKEST PICKS (BOTTOM 3)

1. Content ID: content_677c622c0bfb
   Score: 0.200
   Action: REFRESH
   Reason: STALE
   Why it's weak: 
      - Content may be evergreen (timeless) but penalized
      - Signal combination may be incorrect

2. Content ID: content_689414059706
   Score: 0.200
   Action: REFRESH
   Reason: STALE
   Why it's weak: 
      - Content may be evergreen (timeless) but penalized
      - Signal combination may be incorrect

3. Content ID: content_5eeba5d398f2
   Score: 0.200
   Action: REFRESH
   Reason: STALE
   Why it's weak: 
      - Content may be evergreen (timeless) but penalized
      - Signal combination may be incorrect


### Fix for Weak Picks

| Issue | Fix |
| :--- | :--- |
| Staleness too aggressive | Reduce penalty for evergreen content |
| CTR position bias | Normalize CTR by position |
| Missing signals | Add content-type signal |
| Over-penalization | Add minimum score floor |

**Next steps:** Add content-type detection to reduce false negatives.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.